# Bayes' Theorem — companion notebook

Runnable companion for [`bayes-theorem.md`](./bayes-theorem.md).

1. Disease-testing example numerically, including the base-rate sweep.
2. Sequential Bayesian updating of a Beta posterior from streaming Bernoulli data.
3. Bayes-factor demo: comparing two hypotheses without computing the evidence.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

rng = np.random.default_rng(0)

## 1. Disease testing — and the base rate fallacy

Sensitivity = $P(+ \mid D)$, specificity = $P(- \mid \neg D)$, prevalence = $P(D)$.

In [ ]:
def posterior_given_positive(prevalence, sensitivity, specificity):
    p_pos_given_d = sensitivity
    p_pos_given_not_d = 1 - specificity
    numer = p_pos_given_d * prevalence
    denom = numer + p_pos_given_not_d * (1 - prevalence)
    return numer / denom

print('Single case from the markdown sheet:')
print(f'  P(D | +) = {posterior_given_positive(0.01, 0.99, 0.95):.4f}')

In [ ]:
# Sweep prevalence to show how dramatically the posterior depends on the base rate
prevalences = np.logspace(-4, -0.3, 200)
fig, ax = plt.subplots(figsize=(8, 4))
for sens, spec in [(0.99, 0.95), (0.99, 0.99), (0.95, 0.99)]:
    posts = [posterior_given_positive(p, sens, spec) for p in prevalences]
    ax.plot(prevalences, posts, label=f'sens={sens}, spec={spec}')
ax.set_xscale('log')
ax.set_xlabel('prevalence  P(D)')
ax.set_ylabel('P(D | positive test)')
ax.set_title('Posterior given a positive test vs. base rate')
ax.grid(True, which='both', alpha=0.3); ax.legend()
plt.show()

Even with a 99% sensitive, 99% specific test, when prevalence is below 1% a positive result still leaves substantial probability that the patient is healthy. **The base rate dominates.**

## 2. Sequential Bayesian updating (Beta–Bernoulli conjugate)

Posterior after observing $k$ heads in $n$ flips with Beta$(\alpha, \beta)$ prior is Beta$(\alpha + k, \beta + n - k)$.

In [ ]:
true_p = 0.7
flips = rng.binomial(1, true_p, size=500)

alpha, beta = 2, 2  # weak prior centered on 0.5
xs = np.linspace(0, 1, 400)

fig, ax = plt.subplots(figsize=(8, 4))
for n in [0, 1, 5, 20, 100, 500]:
    k = flips[:n].sum()
    a_post, b_post = alpha + k, beta + n - k
    ax.plot(xs, stats.beta.pdf(xs, a_post, b_post), label=f'n={n}, k={k}')
ax.axvline(true_p, color='black', ls='--', label=f'true p = {true_p}')
ax.set_xlabel('p'); ax.set_ylabel('posterior density')
ax.set_title('Posterior concentrates on the truth as data accumulates')
ax.legend()
plt.show()

## 3. Bayes-factor form — comparing two hypotheses

When you only care about the *ratio* of posteriors, the evidence $p(x)$ cancels out:

$$ \frac{P(H_1 \mid x)}{P(H_2 \mid x)} = \underbrace{\frac{p(x \mid H_1)}{p(x \mid H_2)}}_{\text{Bayes factor}} \cdot \frac{P(H_1)}{P(H_2)} $$

In [ ]:
# Two competing models for the coin: fair (p=0.5) vs biased (p=0.7)
n, k = 20, 14

lik_fair = stats.binom.pmf(k, n=n, p=0.5)
lik_biased = stats.binom.pmf(k, n=n, p=0.7)
bf = lik_biased / lik_fair

print(f'Observed {k}/{n} heads.')
print(f'  L(data | fair)   = {lik_fair:.4f}')
print(f'  L(data | biased) = {lik_biased:.4f}')
print(f'  Bayes factor (biased vs fair) = {bf:.2f}')

# Starting from equal prior odds, what are the posterior odds?
prior_odds = 1.0
post_odds = bf * prior_odds
post_prob_biased = post_odds / (1 + post_odds)
print(f'  Posterior P(biased) = {post_prob_biased:.4f}')